In [4]:
import pandas as pd
import numpy as np
import dagshub
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
# from sentence_transformers import SentenceTransformer
from scipy.sparse import hstack, csr_matrix
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split


In [3]:
pip install dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [5]:
import pandas as pd

# Paste the path you copied from the folder menu here
file_path = '/content/drive/MyDrive/Sentiment analysis /cleaned_data3.csv'

# Load the data into the 'df' variable
df = pd.read_csv(file_path)
df.dropna(inplace = True)
df["Sentiment"] = df["Sentiment"].astype("int8")

In [ ]:
print(df['Sentiment'].value_counts())

Sentiment
-1    316555
 1    316538
 0    316528
Name: count, dtype: int64


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 971460 entries, 0 to 971512
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   CommentText  971460 non-null  object
 1   Sentiment    971460 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 22.2+ MB


In [ ]:
comments = df["CommentText"].tolist()
model = SentenceTransformer("all-MiniLM-L6-v2")

# GPU Encoding: Fast, simple, and has a native progress bar
comments_embeddings = model.encode(
    comments,
    batch_size=256,          # Bump this up from 64 to 256 to feed the GPU faster
    show_progress_bar=True,  # This works perfectly here
    convert_to_numpy=True
).astype(np.float32)

# Save to disk


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3795 [00:00<?, ?it/s]

In [ ]:
comments_embeddings.shape

NameError: name 'comments_embeddings' is not defined

# New Section

In [ ]:
np.save("/content/drive/MyDrive/Sentiment analysis /comments_embeddings1.npy", comments_embeddings)

In [ ]:
df

,CommentText,Sentiment
0,anyone know movie is?,0.0
1,fact theyre hold back equally most aggressive,1.0
2,wait next video be?,0.0
3,thank great video. dont understand db continue...,0.0
4,good person help good people. america exceptio...,1.0
...,...,...
54206,convert future rid right,1.0
54207,movie sonic,0.0
54208,sir installation didnt find android folder cau...,-1.0
54209,long whatinamethetwiceimpeached voters stay ho...,1.0


In [ ]:
loaded_embeddings = np.load("/content/drive/MyDrive/Sentiment analysis /comments_embeddings1.npy")

In [11]:
dagshub.init(repo_owner= 'vaibhav-patel01' , repo_name= 'YT_Comments_Analysis_chrome_extension' , mlflow= True)

mlflow.set_tracking_uri("https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow")
mlflow.set_experiment("finding_best_feature_engineering_algo")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d927404b-2608-41e5-a86c-7b90beccba40&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=7186b3c5190bd02b377d8f516c691c4472b9cbf8be852211ae74ae40f94c23ce




Accessing as vaibhav-patel01

Initialized MLflow to track repo "vaibhav-patel01/YT_Comments_Analysis_chrome_extension"

Repository vaibhav-patel01/YT_Comments_Analysis_chrome_extension initialized!

<Experiment: artifact_location='mlflow-artifacts:/cf5821c3c16a404c81ff005e5f1f60f0', creation_time=1783399519152, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783399519152, lifecycle_stage='active', name='finding_best_feature_engineering_algo', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [ ]:
max_features = 5000
ngrams = (1,3)

In [ ]:
vectorizer = TfidfVectorizer(ngram_range= ngrams, max_features= max_features)
x = vectorizer.fit_transform(df["CommentText"])

y = df["Sentiment"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tags({
            "mlflow.runName" : "selecting_best Max_features",
            "experiment_type" : "feature_engineering",
            "model_type" : "RandomForestClassifier"

            })
        n_estimators = 200
        max_depth = 15

        # Log vectorizer parameters
        mlflow.log_params({
            "vectorizer_type" : "tfidf",
            "n_estimators" : n_estimators,
            "max_depth" : max_depth,
            "max_features" : max_features,
            "n_grams" : ngrams

            })

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)
        model.fit(x_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(x_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        fig = px.imshow(
            conf_matrix,
            text_auto=True, # Equivalent to annot=True
            color_continuous_scale="Blues", # Equivalent to cmap="Blues"
            labels=dict(x="Predicted", y="Actual"),
            title= "Confusion Matrix:",
            width=800, # Approximate equivalent to figsize=(8, 6)
            height=600
        )
        mlflow.log_figure(fig, "basline_confusion_matrix.html")

🏃 View run selecting_best Max_features at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/2/runs/d761ddeb4e064ea3b0e24d9abc27d9cd
🧪 View experiment at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/2


# New Section

In [6]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [7]:
mlflow.set_experiment("Handling Imbalanced Data")

# Step 1: Function to run the experiment
def run_imbalanced_experiment(imbalance_method):
    ngram_range = (1, 3) # Trigram setting
    max_features = 5000 # Set max_features to 1000 for TF-IDF

    # Step 2: Vectorization using TF-IDF
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    X = vectorizer.fit_transform(df['CommentText'])
    y = df['Sentiment']

    # Step 3: Handle class imbalance based on the selected method
    if imbalance_method == 'class_weights':
        # Use class_weight in Random Forest
        class_weight = 'balanced'
    else:
        class_weight = None # Do not apply class_weight if using resampling

    # Resampling Techniques
    if imbalance_method == 'oversampling':
        smote = SMOTE(random_state=42)
        X, y = smote.fit_resample(X, y)
    elif imbalance_method == 'adasyn':
        adasyn = ADASYN(random_state=42)
        X, y = adasyn.fit_resample(X, y)
    elif imbalance_method == 'undersampling':
        rus = RandomUnderSampler(random_state=42)
        X, y = rus.fit_resample(X, y)
    elif imbalance_method == 'smote_enn':
        smote_enn = SMOTEENN(random_state=42)
        X, y = smote_enn.fit_resample(X, y)

    # Step 4: Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Step 5: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"Imbalance_{imbalance_method}_RandomForest_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "imbalance_handling")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, imbalance handling method={imbalance_method}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("imbalance_method", imbalance_method)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, class_weight=class_weight)
        model.fit(X_train, y_train)

        # Step 6: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, Imbalance={imbalance_method}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

# Step 7: Run experiments for different imbalance methods
imbalance_methods = ['adasyn']

for method in imbalance_methods:
    run_imbalanced_experiment(method)

2026/07/09 04:03:35 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/09 04:03:35 INFO mlflow.store.db.utils: Updating database tables
2026/07/09 04:03:37 INFO mlflow.tracking.fluent: Experiment with name 'Handling Imbalanced Data' does not exist. Creating a new experiment.


KeyboardInterrupt: 

In [9]:
pip install optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 10.3 MB/s eta 0:00:00


In [12]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [13]:
mlflow.set_experiment("selecting best algo")

<Experiment: artifact_location='mlflow-artifacts:/8c2385d59dbe485781b9535aaf601075', creation_time=1783570722396, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1783570722396, lifecycle_stage='active', name='selecting best algo', tags={}, trace_location=None, workspace='default'>

In [ ]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

mlflow.set_experiment("selecting best algo")
# Step 1: Remap the class labels from [-1, 0, 1] to [0,1,2]
df['Sentiment'] = df['Sentiment'].map({-1: 0, 0: 1, 1: 2})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['Sentiment'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 5000   # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['CommentText'])
y = df['Sentiment']

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")


# Step 6: Optuna objective function for XGBoost
def objective_xgboost(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = XGBClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


# Step 7: Run Optuna for XGBoost, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_xgboost, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = XGBClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42)

    # Log the best model with MLflow, passing the algo_name as "xgboost"
    log_mlflow("XGBoost", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for XGBoost
run_optuna_experiment()

[I 2026-07-09 04:22:01,737] A new study created in memory with name: no-name-16a5689a-9153-4d37-8dd5-978d7555d51d
[I 2026-07-09 04:46:33,266] Trial 0 finished with value: 0.4868974595234961 and parameters: {'n_estimators': 148, 'learning_rate': 0.0006375755182382179, 'max_depth': 7}. Best is trial 0 with value: 0.4868974595234961.
[I 2026-07-09 04:50:31,966] Trial 1 finished with value: 0.48115308674476764 and parameters: {'n_estimators': 81, 'learning_rate': 0.012336054638939548, 'max_depth': 4}. Best is trial 0 with value: 0.4868974595234961.
[I 2026-07-09 05:18:18,800] Trial 2 finished with value: 0.4842490456759247 and parameters: {'n_estimators': 248, 'learning_rate': 0.00022059424213188378, 'max_depth': 6}. Best is trial 0 with value: 0.4868974595234961.
[I 2026-07-09 05:48:02,077] Trial 3 finished with value: 0.4967171251809925 and parameters: {'n_estimators': 90, 'learning_rate': 0.0008680589003602919, 'max_depth': 9}. Best is trial 3 with value: 0.4967171251809925.
[I 2026-07-